# Model semiparametryczny – zbiór danych Heart Failure

Model Coxa proporcjonalnych hazardów (CoxPH) oraz model Coxa ze zmiennymi zależnymi od czasu dla pacjentów z niewydolnością serca.

- **Zmienna czasu**: `time` (dni)
- **Zdarzenie**: `DEATH_EVENT = 1` (zgon)
- **N = 299** obserwacji; 96 zgonów (32,1 %)

In [ ]:
#!pip install lifelines pandas matplotlib numpy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from lifelines import CoxPHFitter, CoxTimeVaryingFitter
from lifelines.utils import to_episodic_format

df = pd.read_csv('heart_failure_clinical_records_dataset.csv')

print('Ksztalt danych:', df.shape)
print()
selected_cols = ['time', 'DEATH_EVENT', 'age', 'anaemia', 'creatinine_phosphokinase',
                 'diabetes', 'ejection_fraction', 'high_blood_pressure',
                 'platelets', 'serum_creatinine', 'serum_sodium', 'sex', 'smoking']
missing = df[selected_cols].isnull().sum()
print('Braki danych:')
display(missing[missing > 0] if missing.any() else pd.Series({'Brak brakow danych': 0}))
print()
display(df[selected_cols].describe().round(3))


## KOD 1 – Model Coxa proporcjonalnych hazardów (pełny)

Model Coxa ma postać:
$$h(t|\mathbf{x}) = h_0(t) \cdot \exp(\boldsymbol{\beta}^\top \mathbf{x})$$

gdzie $h_0(t)$ to niespecyfikowany hazard bazowy. Parametry $\exp(\beta_j)$ interpretujemy jako **współczynniki ryzyka** (hazard ratio, HR): HR > 1 oznacza zwiększone ryzyko zgonu, HR < 1 – zmniejszone.

In [ ]:
variables = [
    'age', 'anaemia', 'creatinine_phosphokinase', 'diabetes',
    'ejection_fraction', 'high_blood_pressure', 'platelets',
    'serum_creatinine', 'serum_sodium', 'sex', 'smoking'
]

cph = CoxPHFitter()
cph.fit(
    df[variables + ['time', 'DEATH_EVENT']],
    duration_col='time', event_col='DEATH_EVENT'
)
cph.print_summary(model='pelen model Cox', decimals=3)


In [ ]:
# Forest plot wspolczynnikow
fig, ax = plt.subplots(figsize=(8, 6))
cph.plot(ax=ax)
ax.set_title('Wspolczynniki modelu Coxa (95% CI)')
ax.axvline(0, color='red', linestyle='--', linewidth=1)
plt.tight_layout()
plt.savefig('hf_cox_coefficients.png', dpi=150)
plt.show()


## KOD 2 – Weryfikacja założenia proporcjonalnych hazardów

Kluczowe założenie modelu Coxa to **proporcjonalność hazardów**: efekt każdego predyktora nie zmienia się w czasie. Test Schoenfelda: $H_0$: hazardy są proporcjonalne (p > 0,05 = brak dowodów na naruszenie).

In [ ]:
# Test proporcjonalnosci hazardow
print('Weryfikacja zalozenia proporcjonalnych hazardow:')
cph.check_assumptions(
    df[variables + ['time', 'DEATH_EVENT']],
    p_value_threshold=0.05,
    show_plots=True
)


## KOD 3 – Model Coxa zredukowany (zmienne istotne statystycznie)

Usuwamy zmienne nieistotne (p > 0,05) i sprawdzamy założenie na modelu zredukowanym. W literaturze klinicznej kluczowymi predyktorami w Heart Failure są: `ejection_fraction`, `serum_creatinine`, `age`, `serum_sodium`.

In [ ]:
sig_vars = ['age', 'ejection_fraction', 'serum_creatinine', 'serum_sodium',
            'anaemia', 'high_blood_pressure']

cph_red = CoxPHFitter()
cph_red.fit(
    df[sig_vars + ['time', 'DEATH_EVENT']],
    duration_col='time', event_col='DEATH_EVENT'
)
cph_red.print_summary(model='model zredukowany', decimals=3)


In [ ]:
# Sprawdzenie zalozen na modelu zredukowanym
print('Weryfikacja zalozenia proporcjonalnych hazardow (model zredukowany):')
_ = cph_red.check_assumptions(
    df[sig_vars + ['time', 'DEATH_EVENT']],
    p_value_threshold=0.05,
    show_plots=True
)


## KOD 4 – Stratyfikacja zmiennej naruszającej proporcjonalność

Jeśli zmienna narusza założenie proporcjonalności hazardów, stosujemy stratyfikację: osobny hazard bazowy $h_{0k}(t)$ dla każdej warstwy $k$, ale wspólne parametry $\boldsymbol{\beta}$.

In [ ]:
# Binaryzacja serum_creatinine na kwartyle (jesli naruszenie zalozenia)
df_strata = df[sig_vars + ['time', 'DEATH_EVENT']].copy()
df_strata['sc_strata'] = pd.qcut(df_strata['serum_creatinine'], q=3,
                                  labels=['niski', 'sredni', 'wysoki'])

cph_strat = CoxPHFitter()
cph_strat.fit(
    df_strata,
    duration_col='time', event_col='DEATH_EVENT',
    strata=['sc_strata']
)
cph_strat.print_summary(model='stratyfikowany serum_creatinine', decimals=3)

print()
_ = cph_strat.check_assumptions(df_strata, p_value_threshold=0.05)


In [ ]:
# Porownanie AIC modeli Cox
comparison = pd.DataFrame({
    'Model': ['Pelny Cox', 'Zredukowany Cox', 'Stratyfikowany Cox'],
    'Partial AIC': [
        round(cph.AIC_partial_, 2),
        round(cph_red.AIC_partial_, 2),
        round(cph_strat.AIC_partial_, 2)
    ],
    'Concordance': [
        round(cph.concordance_index_, 4),
        round(cph_red.concordance_index_, 4),
        round(cph_strat.concordance_index_, 4)
    ]
})
print('Porownanie modeli Coxa:')
display(comparison)


## KOD 5 – Model Coxa ze zmiennymi zależnymi od czasu

Jeśli zmienna narusza proporcjonalność, możemy wprowadzić **interakcję z czasem** (`zmienna × czas`), co modeluje zmianę efektu w czasie. Dane są przekształcane do formatu episodycznego (start/stop).

In [ ]:
# Przeksztalcenie do formatu episodycznego
df_time = df[sig_vars + ['time', 'DEATH_EVENT']].copy().reset_index()
df_time = df_time.rename(columns={'index': 'id'})

df_long = to_episodic_format(
    df_time,
    duration_col='time',
    event_col='DEATH_EVENT',
    time_gaps=30.0  # okna 30-dniowe
)

print(f'Liczba epizodow: {len(df_long)}')
display(df_long.head(20))


In [ ]:
# Dodanie interakcji czas x serum_creatinine
df_long['time_x_sc'] = df_long['serum_creatinine'] * df_long['stop']

ctv = CoxTimeVaryingFitter()
ctv.fit(
    df_long,
    id_col='id',
    event_col='DEATH_EVENT',
    start_col='start',
    stop_col='stop'
)
ctv.print_summary(5, model='Cox z interakcja czas x serum_creatinine')


In [ ]:
# Wykres krzywej przezyycia wg modelu Coxa dla mediany danych
median_patient = df[sig_vars].median().to_frame().T

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

cph_red.predict_survival_function(median_patient).plot(ax=axes[0])
axes[0].set_title('Funkcja przezycia Coxa – medianowy pacjent')
axes[0].set_xlabel('Czas (dni)')
axes[0].set_ylabel('S(t)')
axes[0].grid(True)

cph_red.predict_cumulative_hazard(median_patient).plot(ax=axes[1], color='red')
axes[1].set_title('Skumulowany hazard Coxa – medianowy pacjent')
axes[1].set_xlabel('Czas (dni)')
axes[1].set_ylabel('H(t)')
axes[1].grid(True)

plt.tight_layout()
plt.savefig('hf_cox_survival_hazard.png', dpi=150)
plt.show()


In [ ]:
# Porownanie przezyycia dla roznych wartosci ejection_fraction
ef_values = [20, 30, 38, 50, 65]  # percentyle + wartosci kliniczne
base_patient = df[sig_vars].median().to_frame().T.copy()

plt.figure(figsize=(10, 6))
for ef in ef_values:
    p = base_patient.copy()
    p['ejection_fraction'] = ef
    sf = cph_red.predict_survival_function(p)
    plt.plot(sf.index, sf.values.flatten(), linewidth=2,
             label=f'EF = {ef}%')

plt.title('Funkcja przezycia Coxa wg frakcji wyrzutowej (EF)\n'
          '(pozostale zmienne = mediana)')
plt.xlabel('Czas (dni)')
plt.ylabel('Prawdopodobienstwo przezycia S(t)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('hf_cox_ef_comparison.png', dpi=150)
plt.show()
